# CrossFuse v5 -- Stage D: Ablations

One seed, reduced epoch budget (8 fine-tune epochs instead of 20), across the
full ablation grid -- including two rows new in v5: `TRIM_LEADING_SILENCE`
(does removing FakeAVCeleb's leading-silence artifact change audio AUC?) and
`SBI_PRETRAIN` (does the SBI-pretrained encoder change video AUC vs. ImageNet
init, holding everything else fixed?).

Writes `/kaggle/working/ablation_results_v5.csv` incrementally, so a crash
partway through loses at most the row in progress. Budget: ~3h for the full
grid on one seed.


In [ ]:
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, copy, json
from collections import defaultdict
import numpy as np
import torch

from crossfuse_v5 import (
    CONFIG, run_training_pipeline, evaluate_deterministic, safe_auc,
    roc_curve, balanced_accuracy_score,
)

_manifest_hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert _manifest_hits, ("manifest.csv not found under /kaggle/input -- attach the "
                        "crops-v5 dataset in Add Input, then restart the session.")
CROP_DIR = os.path.dirname(_manifest_hits[0])
with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    MANIFEST = list(csv.DictReader(f))
for r in MANIFEST:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])

SBI_CKPT_CANDIDATES = glob.glob("/kaggle/input/**/sbi_visual_encoder_v5.pth", recursive=True)
SBI_CKPT = SBI_CKPT_CANDIDATES[0] if SBI_CKPT_CANDIDATES else None

N_ABLATION_SEEDS = 1
ABLATION_EPOCHS = dict(EPOCHS_FROZEN=2, EPOCHS_FINETUNE=8, PATIENCE=4)


def cfg_variant(**overrides):
    c = copy.deepcopy(CONFIG)
    c.update(ABLATION_EPOCHS)
    c.update(overrides)
    return c


def quick_youden(y_true, y_prob, default=0.5):
    y_true = np.asarray(y_true)
    if len(set(y_true.astype(int).tolist())) < 2:
        return default
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])


def quick_eval_row(result, cfg):
    val = evaluate_deterministic(result["model"], result["val_loader"], cfg)
    test = evaluate_deterministic(result["model"], result["test_loader"], cfg)
    row = {"val_video_auc": val["video_auc"], "val_audio_auc": val["audio_auc"],
          "val_sync_auc": val["sync_auc"], "val_fusion_auc": val["fusion_auc"],
          "test_video_auc": test["video_auc"], "test_audio_auc": test["audio_auc"],
          "test_sync_auc": test["sync_auc"], "test_fusion_auc": test["fusion_auc"],
          "n_test": len(result["test_idx"])}
    for head, k in (("video", "v"), ("audio", "a"), ("fusion", "f"), ("sync", "s")):
        vt, vp = val["raw"].get(f"{k}_true"), val["raw"].get(f"{k}_prob")
        tt, tp = test["raw"].get(f"{k}_true"), test["raw"].get(f"{k}_prob")
        if not vt or not tt:
            row[f"{head}_bacc"] = float("nan")
            continue
        thr = quick_youden(vt, vp)
        pred = (np.asarray(tp) >= thr).astype(int)
        row[f"{head}_bacc"] = (balanced_accuracy_score(tt, pred)
                               if len(set(np.asarray(tt).astype(int).tolist())) > 1
                               else float("nan"))
    return row


In [ ]:
ABLATIONS = [
    ("A1_video_only", cfg_variant(MODALITY="video_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False)),
    ("A2_audio_only", cfg_variant(MODALITY="audio_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False)),
    ("A3_two_head_no_fusion", cfg_variant(USE_CROSS_ATTENTION=False, USE_SYNC_HEAD=False,
                                          USE_FUSION_HEAD=False)),
    ("A4_full_crossfuse", cfg_variant()),   # matches NB-B's main config
    ("A5_no_hard_negatives", cfg_variant(GENERATE_HARD_NEGATIVES=False)),
    ("A6_no_compression_aug", cfg_variant(USE_COMPRESSION_AUG=False)),
    ("A7_no_mfcc_cmvn", cfg_variant(USE_MFCC_CMVN=False)),
    ("A8_random_split", cfg_variant(IDENTITY_DISJOINT_SPLIT=False)),
    ("A9_late_fusion_baseline", cfg_variant(LATE_FUSION_BASELINE=True)),
    ("A10_sync_unaligned", cfg_variant(SYNC_DENSE_ALIGNED=False)),
    ("A11_no_paired_batches", cfg_variant(USE_PAIRED_BATCHES=False)),
    # New in v5:
    ("A12_no_silence_trim", cfg_variant(AUDIO_LEAD_SKIP=0.0)),
    ("A13_sbi_pretrained", cfg_variant(SBI_PRETRAINED_ENCODER=SBI_CKPT)) if SBI_CKPT else None,
]
ABLATIONS = [a for a in ABLATIONS if a is not None]
if SBI_CKPT is None:
    print("[i] No SBI checkpoint attached -- skipping A13_sbi_pretrained. "
          "Run NB-B0 and attach 'sbi-visual-encoder-v5' to include it.")

print(f"Running {len(ABLATIONS)} ablation configs x {N_ABLATION_SEEDS} seed(s)...")


In [ ]:
ablation_csv_path = "/kaggle/working/ablation_results_v5.csv"
fieldnames = ["row", "seed", "n_test", "val_video_auc", "val_audio_auc", "val_sync_auc",
             "val_fusion_auc", "test_video_auc", "test_audio_auc", "test_sync_auc",
             "test_fusion_auc", "video_bacc", "audio_bacc", "fusion_bacc", "sync_bacc"]


def _write_ablation_csv(rows):
    with open(ablation_csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


ablation_rows = []
for name, cfg in ABLATIONS:
    for si in range(N_ABLATION_SEEDS):
        seed = CONFIG["SEED"] + si
        print(f"\n{'=' * 60}\n  {name}  (seed {seed})\n{'=' * 60}")
        result = run_training_pipeline(MANIFEST, CROP_DIR, cfg, tag=name, seed=seed,
                                       verbose=True)
        row = quick_eval_row(result, cfg)
        row["row"] = name
        row["seed"] = seed
        ablation_rows.append(row)
        _write_ablation_csv(ablation_rows)
        print(f"{name} (seed {seed}) -> video={row['test_video_auc']:.4f} "
              f"audio={row['test_audio_auc']:.4f} fusion={row['test_fusion_auc']:.4f} "
              f"sync={row['test_sync_auc']:.4f}")
        del result["model"]
        torch.cuda.empty_cache()

print(f"\nAll ablations complete. Written to {ablation_csv_path}")
